In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open("../names.txt").read().splitlines()
len(words)

32033

In [32]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
encode  = lambda s: [stoi[char] for char in s.lower()]
decode = lambda x: ''.join(itos[ix] for ix in x)


In [33]:
encode("Soumyajit")
decode(encode("Soumyajit"))

'soumyajit'

In [34]:
block_size = 3

def build_dataset(words):
    X = []
    Y = []
    
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            conetext = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(f"{X.shape=} {Y.shape=}")

    return X, Y

x, y = build_dataset(["anam", "balle"])

X.shape=torch.Size([11, 3]) Y.shape=torch.Size([11])


In [36]:
import random 

random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xval, Yval = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

X.shape=torch.Size([182625, 3]) Y.shape=torch.Size([182625])
X.shape=torch.Size([22655, 3]) Y.shape=torch.Size([22655])
X.shape=torch.Size([22866, 3]) Y.shape=torch.Size([22866])


In [64]:
# MLP classes
g = torch.Generator().manual_seed(2147483647) # for reproducibility

class Linear:

    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g)
        self.bias = torch.randn((fan_out, ), generator=g) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

lin = Linear(4, 6, False)
assert len(lin.parameters()) == 1

In [ ]:
class Batchnorm:

    def __init__(self, dim, eps=1e-5, momentum=0.1, train=True):
        self.eps = eps
        self.momentum = momentum
        self.training = train

        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            xmean = x.mean(axis=0, keepdims=True)
            xvar = x.var(axis=0, keepdims=True)
        else:
            xmean = self.running_mean 
            xvar = self.running_var

        xhat = (x - xmean) / torch.sqrt(xvar + eps)
        self.out = xhat * self.gamma + self.beta
        if self.trainig:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentun * xmean
                self.running_var =  (1 - self.momentun) * self.running_var + self.momentun * xvar

        return self.out

    def parameters(self, ):
        return [self.gamma, self.beta]


class Tanh:

    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self, x):
        return []